# License
This notebook is licensed under the MIT License. See the LICENSE file in the repository root for details.

In [ ]:
# Setup
try:
    import google.colab
    IN_COLAB = True
    !pip install -q pandas numpy matplotlib scipy
except ImportError:
    IN_COLAB = False

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# Set seed for reproducibility
np.random.seed(42)

# Synthetic Control Methods (SCM)

The **Synthetic Control Method (SCM)** is a statistical method used to evaluate the effect of an intervention in comparative case studies. It involves constructing a weighted combination of control units (the "synthetic control") that best resembles the characteristics of the treated unit prior to the intervention.

## Scenario
Imagine a city ("City A") implements a new tax policy in year 2010. We want to know the effect of this tax on **GDP per capita**. We have data for City A and 10 other cities that did *not* implement the tax.

Instead of just comparing City A to the average of the other cities, we create a "Synthetic City A" which is a weighted average of the other cities, chosen to match City A's GDP trajectory *before* 2010.


## 1. Generating Synthetic Data

We simulate panel data for 11 cities (1 treated, 10 controls) over 20 years (2000-2020).
*   **Intervention**: Year 2010.
*   **Effect**: The policy causes a gradual decrease in GDP.


In [ ]:
years = np.arange(2000, 2021)
n_years = len(years)
intervention_year = 2010
intervention_index = np.where(years == intervention_year)[0][0]

# Generate Control Units (Cities B-K)
# They follow a random walk with drift
controls = []
names = [f"City {chr(66+i)}" for i in range(10)] # City B to K

for _ in range(10):
    start_gdp = np.random.normal(30000, 5000)
    growth = np.random.normal(500, 100, n_years)
    gdp = np.cumsum(np.insert(growth, 0, start_gdp))[:-1]
    controls.append(gdp)

controls_df = pd.DataFrame(np.array(controls).T, columns=names, index=years)

# Generate Treated Unit (City A)
# Constructed as a weighted average of controls + noise + treatment effect
true_weights = np.array([0.4, 0.3, 0.2, 0.1] + [0]*6) # It mostly resembles first 4 cities
base_gdp = np.dot(controls_df.values, true_weights)
noise = np.random.normal(0, 200, n_years)

# Treatment effect: 0 before 2010, negative after
effect = np.zeros(n_years)
post_indices = np.where(years > intervention_year)[0]
for i, idx in enumerate(post_indices):
    effect[idx] = -500 * (i + 1) # Cumulative decline

city_a_gdp = base_gdp + noise + effect
controls_df['City A'] = city_a_gdp

print("Data generated. Head:")
print(controls_df.head())

# Plot all cities
plt.figure(figsize=(10, 6))
plt.plot(controls_df.index, controls_df.drop('City A', axis=1), color='gray', alpha=0.5, label='Controls')
plt.plot(controls_df.index, controls_df['City A'], color='blue', linewidth=3, label='City A (Treated)')
plt.axvline(intervention_year, linestyle='--', color='black', label='Intervention')
plt.title("GDP per Capita: Treated vs Controls")
plt.legend(['Controls'] + ['City A', 'Intervention']) # Simplify legend
plt.show()


## 2. Finding the Weights

We need to find weights $W = (w_1, ..., w_{10})$ such that:
1.  $\sum w_i = 1$
2.  $w_i \ge 0$
3.  The weighted average of controls matches City A as closely as possible *before* the intervention.

We minimize the Mean Squared Error (MSE) in the pre-intervention period.


In [ ]:
# Split into pre-intervention data
X_treated = controls_df.loc[years <= intervention_year, 'City A'].values
X_controls = controls_df.loc[years <= intervention_year, names].values

# Loss function
def loss_function(weights, treated, controls):
    synthetic = np.dot(controls, weights)
    return np.mean((treated - synthetic)**2)

# Constraints and Bounds
n_controls = len(names)
constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1}) # Sum to 1
bounds = [(0, 1) for _ in range(n_controls)] # Non-negative

# Initial guess (uniform weights)
init_weights = [1/n_controls] * n_controls

# Optimize
result = minimize(loss_function, 
                  init_weights, 
                  args=(X_treated, X_controls),
                  method='SLSQP', 
                  bounds=bounds, 
                  constraints=constraints)

best_weights = result.x

print("Optimization Success:", result.success)
print("Best Weights:")
for name, w in zip(names, best_weights):
    if w > 0.001: # Filter small weights
        print(f"{name}: {w:.3f}")


## 3. Visualizing the Result

Now we construct the "Synthetic City A" for the *entire* period using the optimal weights and compare it to the real City A.


In [ ]:
# Construct Synthetic City A for all years
all_controls = controls_df[names].values
synthetic_city_a = np.dot(all_controls, best_weights)

plt.figure(figsize=(12, 6))
plt.plot(years, controls_df['City A'], label='Real City A', color='blue', linewidth=2)
plt.plot(years, synthetic_city_a, label='Synthetic City A', color='red', linestyle='--', linewidth=2)
plt.axvline(intervention_year, color='black', linestyle=':', label='Intervention (2010)')

# Highlight the gap
plt.fill_between(years, controls_df['City A'], synthetic_city_a, where=(years > intervention_year), color='gray', alpha=0.3, label='Treatment Effect')

plt.title("Synthetic Control Method: Impact of Tax Policy")
plt.xlabel("Year")
plt.ylabel("GDP per Capita")
plt.legend()
plt.show()


## 4. Estimating the Effect Size

The causal effect is the difference between the observed outcome and the synthetic counterfactual.


In [ ]:
gap = controls_df['City A'] - synthetic_city_a

plt.figure(figsize=(12, 4))
plt.plot(years, gap, marker='o', color='purple')
plt.axhline(0, color='black', linestyle='-')
plt.axvline(intervention_year, color='black', linestyle=':')
plt.title("Gap (Real - Synthetic)")
plt.ylabel("Difference in GDP")
plt.show()

avg_effect = gap[years > intervention_year].mean()
print(f"Average estimated effect after 2010: ${avg_effect:.2f}")
